# Voicebox GPU worker

Turns this Colab runtime into the voice engine behind your **voicebox MCP** (Kokoro, Chatterbox, Orpheus, Dia).

**Once:** open the 🔑 *Secrets* panel (left sidebar), add and enable notebook access for:

| Secret | Value |
|---|---|
| `WORKER_SECRET` | the `WORKER_SECRET` line from `.secrets.local` |
| `REGISTER_URL` | `https://<your-vercel-app>/api/register` |
| `PREWARM` *(optional)* | engines to load at start, e.g. `orpheus,whisper` |

**Every session:** Runtime → Change runtime type → **L4** (or A100) → *Run all*. The last cell stays busy while the worker serves;
stop it (or disconnect the runtime) when you're done to save compute units.

In [ ]:
# 1 · GPU check + secrets + worker code (latest from your server; embedded v3.15.2 as fallback)
import base64, io, os, subprocess, tarfile, urllib.request
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "No GPU! Runtime → Change runtime type → L4")
from google.colab import userdata
for k in ("WORKER_SECRET", "REGISTER_URL", "PREWARM", "VOICEBOX_VRAM_GB"):
    try:
        v = userdata.get(k)
        if v: os.environ[k] = v
    except Exception:
        pass  # optional secrets may be missing
assert len(os.environ.get("WORKER_SECRET", "")) >= 24, "Add WORKER_SECRET in Secrets (🔑) and allow notebook access"
APP = "/content/voicebox/app"
os.makedirs(APP, exist_ok=True)
BUNDLE = "..."
data, source = base64.b64decode(BUNDLE), "embedded v3.15.2"
if os.environ.get("REGISTER_URL"):
    try:
        req = urllib.request.Request(os.environ["REGISTER_URL"].rsplit("/api/", 1)[0] + "/api/bundle",
                                     headers={"Authorization": "Bearer " + os.environ["WORKER_SECRET"], "User-Agent": "voicebox-worker"})
        with urllib.request.urlopen(req, timeout=30) as r:
            data, source = r.read(), "server v" + r.headers.get("X-Worker-Version", "?")
    except Exception as e:
        print("server bundle unavailable, using embedded:", e)
tarfile.open(fileobj=io.BytesIO(data), mode="r:gz").extractall(APP, filter="data")
print("worker", source, "→", APP)

In [ ]:
# 2 · Start: gateway + Cloudflare tunnel + heartbeat to the MCP server. Keeps running.
import os, runpy, sys
if not os.environ.get("REGISTER_URL"): print("⚠ REGISTER_URL not set — the dashboard won't find this worker")
sys.argv = ["launcher.py"]
runpy.run_path("/content/voicebox/app/launcher.py", run_name="__main__")

In [ ]:
# (debug) tail the logs — run this in a *second* tab/scratch cell while the worker runs
!tail -n 40 /content/voicebox/logs/gateway.log; ls /content/voicebox/logs